# ALGS Year 4 版本运营数据分析：可复现 Notebook

## 0. Project README / Run instructions

这个 notebook 用来复现作品集报告中的核心数据表和图表。它不是代码草稿，而是一份可以顺着阅读、检查、复跑的数据分析记录。

**项目目标**

- 复现报告中的 5 张核心 dashboard：总分与 reset 信号、阵容集中度、武器生态迁移、角色职业槽位、队伍打法 proxy。
- 保留数据来源、数据校验、指标计算、图表生成和输出导出流程。
- 让读者能对应到报告里的图表、表格和结论。

**公开数据来源**

- `data/portfolio_algs_year4/`：从公开赛事统计页面抓取并整理的 ALGS Year 4 四次国际赛事数据，主要来自 Apex Legends Status 的 ALGS 赛事统计页面。
- `patch_notes_items.csv` / `patch_notes_sources.csv` / `event_patch_context.csv`：用于报告第一层的版本更新与赛事窗口对应关系。
- 本 notebook 不会修改原始数据，所有清洗、指标和图表都会输出到 `portfolio/notebooks/outputs/`。

**数据限制**

公开表格足够支持“得分结构、阵容使用、角色职业、武器击杀、复活/救起信号”的分析，但它不包含完整路线、坐标、圈型、每次交战事件和队伍实时决策。因此，报告中的队伍打法分类是基于击杀分占比与 Top 5 转化率构造的 **proxy**，不是对每支队伍真实路线的还原。

**运行方式**

1. 在项目根目录或 `portfolio/notebooks/` 下打开本 notebook。
2. 依次运行全部单元格。
3. 输出会写入：
   - `outputs/tables/`：报告表格和关键指标 CSV。
   - `outputs/figures/`：报告图表 PNG。
   - `outputs/intermediate/`：清洗后的中间结果。

**报告对应关系**

- Figure 1 对应报告“图表 1：总击杀稳定，但 reset 信号上升”。
- Figure 2 对应报告“图表 2：阵容集中度先被打散，又重新收敛”。
- Figure 3 对应报告“图表 3：武器生态发生了三次迁移”。
- Figure 4 对应报告“图表 4：角色职业槽位暴露版本需求”。
- Figure 5 对应报告“图表 5：纯圈边击杀不是最高收益”。
- Section 6 risk flags 对应报告第三层/第四层使用的运营监控口径。


## 1. Load data

这一节只做两件事：定位项目目录、读取原始 CSV。复杂计算全部放到后面的指标函数中。

下表说明每个原始文件在报告里承担的作用。

| 文件 | 包含什么 | 报告用途 |
|---|---|---|
| `events.csv` | 四次赛事的基础信息和页面入口 | 赛事范围、公开来源说明 |
| `event_games.csv` | 每局比赛的 game_id、地图、stage、URL | 局数校验、地图池校验 |
| `event_context.csv` | 赛事名称、日期、赛制、地图池等整理信息 | 报告开头与第一层背景 |
| `event_patch_context.csv` | 赛事窗口与版本更新之间的对应关系 | 第一层“版本想改变什么” |
| `patch_notes_sources.csv` | patch notes 来源链接 | 第一层版本信息来源 |
| `patch_notes_items.csv` | 版本更新条目结构化摘录 | 第一层设计信号分析 |
| `post_championship_year5_updates.csv` | Championship 后续 Year 5 规则/版本回应 | 第四层现实验证 |
| `match_scores.csv` | 每局每队排名分、击杀分、总分 | Figure 1、打法 proxy、基础得分校验 |
| `game_team_stats.csv` | 每局每队详细表现，如 knocks、deaths、rez、rspn | Figure 1 reset 信号、风险标志 |
| `overview_team_stats.csv` | 每次赛事每队汇总表现 | Figure 5 队伍打法 proxy |
| `overview_composition_meta.csv` | 每次赛事阵容组合 pick rate、Top 5、胜率等 | Figure 2 阵容集中度 |
| `game_composition_by_scope.csv` | 每局阵容组合记录 | 数据完整性校验 |
| `game_composition_meta.csv` | 每局阵容聚合指标 | 数据追溯和局内阵容检查 |
| `overview_composition_by_scope.csv` | overview 层级阵容记录 | 阵容统计来源交叉检查 |
| `overview_legend_meta.csv` | 每次赛事单角色 pick rate、Top 5、胜率等 | Figure 4 职业槽位、风险标志 |
| `game_legend_by_scope.csv` | 每局每角色出场记录 | 阵容是否 3 legend 的辅助校验 |
| `game_legend_meta.csv` | 每局角色聚合指标 | 局内角色使用追溯 |
| `overview_legend_by_scope.csv` | overview 层级角色记录 | 角色统计来源交叉检查 |
| `weapon_stats.csv` | overview 与逐局武器 kills、damage、knockdowns 等 | Figure 3 武器迁移、单武器榜 |
| `overview_player_stats.csv` | 每次赛事玩家汇总表现 | 补充分析，不直接生成本报告核心图 |
| `game_player_stats.csv` | 每局玩家表现 | 补充分析，不直接生成本报告核心图 |
| `overview_team_map_stats.csv` | 队伍在不同地图上的汇总表现 | 地图维度补充分析 |
| `game_team_map_stats.csv` | 单局队伍地图表现 | 地图维度补充校验 |
| `overview_poi_stats.csv` | overview 层级 POI 统计 | 选点/资源补充材料 |
| `game_poi_stats.csv` | 单局 POI 统计 | 选点/资源补充材料 |
| `overview_other.csv` | 页面中未归入主表的 overview 数据 | 原始抓取留档 |
| `game_other.csv` | 页面中未归入主表的单局数据 | 原始抓取留档 |
| `overview_raw_table_rows.csv` | overview 原始表格行 | 抓取审计留档 |
| `game_raw_table_rows.csv` | 单局原始表格行 | 抓取审计留档 |


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import random
import textwrap

import pandas as pd
from PIL import Image, ImageDraw, ImageFont

random.seed(42)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

try:
    from IPython.display import display
except Exception:
    def display(obj):
        if hasattr(obj, 'to_string'):
            print(obj.to_string(index=False))
        else:
            print(obj)


def find_data_dir() -> tuple[Path, Path, Path]:
    """Find raw data and output directories whether run from repo or package."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for base in candidates:
        raw_repo = base / 'data' / 'portfolio_algs_year4'
        if raw_repo.exists():
            return base, raw_repo, base / 'portfolio' / 'notebooks' / 'outputs'
        raw_package = base / 'data' / 'raw' / 'portfolio_algs_year4'
        if raw_package.exists():
            return base, raw_package, base / 'notebook' / 'outputs'
    raise FileNotFoundError('Cannot find data/portfolio_algs_year4 or data/raw/portfolio_algs_year4')

ROOT, DATA_DIR, OUTPUT_DIR = find_data_dir()
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
INTERMEDIATE_DIR = OUTPUT_DIR / 'intermediate'
for directory in [TABLE_DIR, FIGURE_DIR, INTERMEDIATE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EVENT_ORDER = ['playoffs_split1', 'midseason_ewc_split1', 'playoffs_split2', 'championship_split2']
EVENT_LABEL = {
    'playoffs_split1': 'Split 1',
    'midseason_ewc_split1': 'EWC',
    'playoffs_split2': 'Split 2',
    'championship_split2': 'Championship',
}
EXPECTED_GAMES = {
    'playoffs_split1': 62,
    'midseason_ewc_split1': 43,
    'playoffs_split2': 64,
    'championship_split2': 69,
}

LEGEND_CLASS_MAPPING = {
    'Ash': 'Assault', 'Ballistic': 'Assault', 'Bangalore': 'Assault', 'Fuse': 'Assault', 'Mad Maggie': 'Assault',
    'Alter': 'Skirmisher', 'Horizon': 'Skirmisher', 'Octane': 'Skirmisher', 'Pathfinder': 'Skirmisher',
    'Revenant': 'Skirmisher', 'Valkyrie': 'Skirmisher', 'Wraith': 'Skirmisher',
    'Bloodhound': 'Recon', 'Crypto': 'Recon', 'Seer': 'Recon', 'Vantage': 'Recon',
    'Catalyst': 'Controller', 'Caustic': 'Controller', 'Rampart': 'Controller', 'Wattson': 'Controller',
    'Conduit': 'Support', 'Gibraltar': 'Support', 'Lifeline': 'Support', 'Loba': 'Support',
    'Mirage': 'Support', 'Newcastle': 'Support',
}

WEAPON_CATEGORY_MAPPING = {
    '30-30 Repeater': 'Precision', 'Bocek Compound Bow': 'Precision', 'Charge Rifle': 'Precision',
    'G7 Scout': 'Precision', 'Kraber .50-Cal Sniper': 'Precision', 'Longbow DMR': 'Precision',
    'Sentinel': 'Precision', 'Triple Take': 'Precision', 'Wingman': 'Precision',
    'HAVOC Rifle': 'AR', 'Hemlok': 'AR', 'Nemesis Burst AR': 'AR', 'R-301 Carbine': 'AR', 'VK-47 Flatline': 'AR',
    'Alternator SMG': 'SMG', 'C.A.R.': 'SMG', 'Prowler Burst PDW': 'SMG', 'R-99 SMG': 'SMG', 'Volt SMG': 'SMG',
    'RE-45': 'Pistol/SMG',
    'Devotion LMG': 'LMG', 'L-STAR EMG': 'LMG', 'M600 Spitfire': 'LMG', 'Rampage LMG': 'LMG',
    'EVA-8 Auto': 'Shotgun', 'Mastiff Shotgun': 'Shotgun', 'Mozambique Shotgun': 'Shotgun', 'Peacekeeper': 'Shotgun',
    'Mozambique (Akimbo)': 'Shotgun-Akimbo', 'P2020': 'Pistol/Akimbo',
    'Arc Star': 'Ordnance', 'Frag Grenade': 'Ordnance', 'Thermite Grenade': 'Ordnance',
    'Fall': 'Other', 'Melee': 'Other',
}

EVENT_DISPLAY_ORDER = [EVENT_LABEL[e] for e in EVENT_ORDER]
print(f'Project root: {ROOT}')
print(f'Data dir:     {DATA_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')


In [ ]:
# Read every raw CSV once and keep them in a dictionary for traceability.
raw_tables: dict[str, pd.DataFrame] = {}
for csv_path in sorted(DATA_DIR.glob('*.csv')):
    table_name = csv_path.stem
    raw_tables[table_name] = pd.read_csv(csv_path, low_memory=False)

loaded_files = pd.DataFrame([
    {'table': name, 'rows': len(df), 'columns': len(df.columns), 'source_file': f'{name}.csv'}
    for name, df in raw_tables.items()
]).sort_values('table')

# Primary source tables used by the figures and risk flags.
match_scores = raw_tables['match_scores'].copy()
game_team_stats = raw_tables['game_team_stats'].copy()
overview_team_stats = raw_tables['overview_team_stats'].copy()
overview_composition_meta = raw_tables['overview_composition_meta'].copy()
game_composition_by_scope = raw_tables['game_composition_by_scope'].copy()
overview_legend_meta = raw_tables['overview_legend_meta'].copy()
weapon_stats = raw_tables['weapon_stats'].copy()

display(loaded_files)


## 2. Data validation

这一节先回答“数据能不能信”。校验分成五类：赛事局数、每局队伍数、总分公式、缺失/重复、阵容是否由 3 个 legend 构成。

注意：逐局阵容表中 EWC 有一局的阵容为空，这是公开页面逐局阵容抓取层面的缺口。报告中的阵容集中度来自 `overview_composition_meta.csv` 的赛事级统计，不直接依赖这 40 行逐局阵容。


In [ ]:
def to_number(series: pd.Series) -> pd.Series:
    """Convert percentage/comma-formatted strings to numeric values."""
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False).str.replace('%', '', regex=False),
        errors='coerce'
    )

validation_rows = []

def add_check(check: str, status: str, details: str, affected_rows: int | None = None) -> None:
    validation_rows.append({
        'check': check,
        'status': status,
        'affected_rows': affected_rows,
        'details': details,
    })

# 1) Event game counts.
actual_games = match_scores.groupby('event_slug')['game_id'].nunique().reindex(EVENT_ORDER)
for event_slug, expected in EXPECTED_GAMES.items():
    actual = int(actual_games.loc[event_slug])
    add_check(
        f'game_count_{EVENT_LABEL[event_slug]}',
        'pass' if actual == expected else 'fail',
        f'expected={expected}, actual={actual}',
        abs(actual - expected),
    )

# 2) Team count per game.
team_rows_per_game = match_scores.groupby(['event_slug', 'game_id']).size().rename('team_rows').reset_index()
bad_team_count = team_rows_per_game[team_rows_per_game['team_rows'] != 20]
add_check(
    'team_rows_per_game',
    'pass' if bad_team_count.empty else 'warn',
    f'min={team_rows_per_game.team_rows.min()}, max={team_rows_per_game.team_rows.max()}, games_not_20={len(bad_team_count)}',
    len(bad_team_count),
)

# 3) Score formula.
score_check_df = match_scores.copy()
for col in ['kills', 'total_points', 'placement_points_reported', 'placement_points_rule']:
    score_check_df[col] = to_number(score_check_df[col])
score_formula_mismatch = score_check_df[
    score_check_df['total_points'] != score_check_df['kills'] + score_check_df['placement_points_reported']
]
score_check_not_ok = score_check_df[score_check_df['score_check'].astype(str).str.lower() != 'ok']
add_check(
    'total_points_formula',
    'pass' if score_formula_mismatch.empty else 'fail',
    'total_points == kills + placement_points_reported',
    len(score_formula_mismatch),
)
add_check(
    'score_check_column',
    'pass' if score_check_not_ok.empty else 'fail',
    'score_check column should be ok for all rows',
    len(score_check_not_ok),
)

# 4) Missing values and duplicate records in important tables.
critical_columns = {
    'match_scores': ['event_slug', 'game_id', 'team', 'placement', 'kills', 'total_points'],
    'game_team_stats': ['event_slug', 'game_id', 'team', 'kills', 'deaths', 'rez', 'rspn'],
    'overview_team_stats': ['event_slug', 'team', 'games', 'kills', 'place_pts', 'total_pts', 'top_5s'],
    'overview_composition_meta': ['event_slug', 'composition', 'pick_rate_pct', 'top_5_rate_pct', 'win_rate_pct'],
    'overview_legend_meta': ['event_slug', 'legend', 'pick_rate_pct', 'top_5_rate_pct'],
    'weapon_stats': ['event_slug', 'source_scope', 'weapon', 'kills', 'damage'],
}
for table_name, columns in critical_columns.items():
    df = raw_tables[table_name]
    critical_missing = int(df[columns].isna().sum().sum())
    duplicate_rows = int(df.duplicated().sum())
    add_check(
        f'{table_name}_critical_missing',
        'pass' if critical_missing == 0 else 'warn',
        f'critical columns checked: {columns}',
        critical_missing,
    )
    add_check(
        f'{table_name}_duplicate_rows',
        'pass' if duplicate_rows == 0 else 'warn',
        'full-row duplicate count',
        duplicate_rows,
    )

# 5) Team compositions should have three legends when present.
composition_check = game_composition_by_scope.copy()
composition_check['legend_count'] = composition_check['composition'].dropna().astype(str).map(
    lambda value: len([part.strip() for part in value.split(',') if part.strip()])
)
composition_missing = int(composition_check['composition'].isna().sum())
bad_legend_count = composition_check[composition_check['composition'].notna() & (composition_check['legend_count'] != 3)]
add_check(
    'team_composition_missing_rows',
    'warn' if composition_missing else 'pass',
    'missing composition rows are retained in the quality report',
    composition_missing,
)
add_check(
    'team_composition_three_legends',
    'pass' if bad_legend_count.empty else 'fail',
    'all non-missing team composition rows should contain exactly 3 legends',
    len(bad_legend_count),
)

# Save key cleaned source tables used downstream.
score_check_df.to_csv(INTERMEDIATE_DIR / 'match_scores_clean.csv', index=False)
composition_check.to_csv(INTERMEDIATE_DIR / 'game_composition_quality_check.csv', index=False)
team_rows_per_game.to_csv(INTERMEDIATE_DIR / 'team_rows_per_game.csv', index=False)

data_quality_summary = pd.DataFrame(validation_rows)
data_quality_summary.to_csv(TABLE_DIR / 'data_quality_summary.csv', index=False)
display(data_quality_summary)


## 3. Core metric functions

这里把重复计算整理成函数。每个函数的 docstring 都写清楚输入、输出字段，以及对应报告里的图表或章节。


In [ ]:
def add_event_label(df: pd.DataFrame, event_col: str = 'event_slug') -> pd.DataFrame:
    result = df.copy()
    order_map = {event_slug: index for index, event_slug in enumerate(EVENT_ORDER)}
    result['_event_order'] = result[event_col].map(order_map).fillna(999)
    result = result.sort_values(['_event_order', event_col]).drop(columns=['_event_order']).reset_index(drop=True)
    result['event_label'] = result[event_col].map(EVENT_LABEL)
    return result


def calc_score_structure(match_scores: pd.DataFrame) -> pd.DataFrame:
    """Calculate event-level score structure.

    Input:
        match_scores: one row per team-game, with event_slug, game_id, team, kills,
        total_points and placement_points_reported.

    Output fields:
        event_slug, event_label, games, team_game_rows, total_kills,
        placement_points, total_points, kills_per_game, kill_point_share_pct.

    Report mapping:
        Figure 1 / 图表 1，用于验证“总击杀和击杀分占比整体稳定”。
    """
    scores = match_scores.copy()
    for col in ['kills', 'total_points', 'placement_points_reported']:
        scores[col] = to_number(scores[col])
    result = scores.groupby('event_slug').agg(
        games=('game_id', 'nunique'),
        team_game_rows=('team', 'count'),
        total_kills=('kills', 'sum'),
        placement_points=('placement_points_reported', 'sum'),
        total_points=('total_points', 'sum'),
    ).reset_index()
    result['kills_per_game'] = result['total_kills'] / result['games']
    result['kill_point_share_pct'] = 100 * result['total_kills'] / result['total_points']
    return add_event_label(result)


def calc_reset_metrics(game_team_stats: pd.DataFrame) -> pd.DataFrame:
    """Calculate event-level reset and second-chance signals.

    Input:
        game_team_stats: one row per team-game, with event_slug, game_id, deaths,
        rez, rspn, knocks and related combat telemetry.

    Output fields:
        event_slug, event_label, games, deaths_per_game, avg_rez, avg_rspn,
        knocks_per_game, deaths_above_60.

    Report mapping:
        Figure 1 / 图表 1，用于验证“总击杀稳定，但 reset 信号上升”。
    """
    stats = game_team_stats.copy()
    for col in ['deaths', 'rez', 'rspn', 'knocks', 'times_knocked', 'ring_dmg']:
        stats[col] = to_number(stats[col])
    per_game = stats.groupby(['event_slug', 'game_id']).agg(
        deaths=('deaths', 'sum'),
        rez=('rez', 'sum'),
        rspn=('rspn', 'sum'),
        knocks=('knocks', 'sum'),
        times_knocked=('times_knocked', 'sum'),
        ring_damage=('ring_dmg', 'sum'),
    ).reset_index()
    result = per_game.groupby('event_slug').agg(
        games=('game_id', 'nunique'),
        deaths_per_game=('deaths', 'mean'),
        avg_rez=('rez', 'mean'),
        avg_rspn=('rspn', 'mean'),
        knocks_per_game=('knocks', 'mean'),
        times_knocked_per_game=('times_knocked', 'mean'),
        ring_damage_per_game=('ring_damage', 'mean'),
    ).reset_index()
    result['deaths_above_60'] = result['deaths_per_game'] - 60
    per_game.to_csv(INTERMEDIATE_DIR / 'reset_metrics_by_game.csv', index=False)
    return add_event_label(result)


def calc_composition_concentration(team_compositions: pd.DataFrame) -> pd.DataFrame:
    """Calculate Top 1 / Top 3 / Top 5 composition concentration.

    Input:
        team_compositions: event-level composition table, usually overview_composition_meta,
        with event_slug, composition and pick_rate_pct.

    Output fields:
        event_slug, event_label, composition_count, top1_composition,
        top1_comp_share, top3_comp_share, top5_comp_share.

    Report mapping:
        Figure 2 / 图表 2，用于说明“阵容集中度先被打散，又重新收敛”。
    """
    comps = team_compositions.copy()
    comps['pick_rate_pct'] = to_number(comps['pick_rate_pct'])
    rows = []
    for event_slug, group in comps.groupby('event_slug'):
        group = group.sort_values('pick_rate_pct', ascending=False).reset_index(drop=True)
        rows.append({
            'event_slug': event_slug,
            'composition_count': len(group),
            'top1_composition': group.loc[0, 'composition'],
            'top1_comp_share': group.head(1)['pick_rate_pct'].sum(),
            'top3_comp_share': group.head(3)['pick_rate_pct'].sum(),
            'top5_comp_share': group.head(5)['pick_rate_pct'].sum(),
        })
    return add_event_label(pd.DataFrame(rows))


def calc_weapon_share(weapon_stats: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate weapon category and single-weapon kill/damage share.

    Input:
        weapon_stats: weapon table with source_scope, event_slug, weapon, kills and damage.
        This function uses overview rows to avoid double-counting per-game rows.

    Output fields:
        category_share: event_slug, event_label, category, kills, damage,
        kill_share_pct, damage_share_pct.
        single_weapon_share: event_slug, event_label, weapon, category, kills, damage,
        kill_share_pct, damage_share_pct.

    Report mapping:
        Figure 3 / 图表 3，用于说明武器生态从 AR 到 Akimbo/Shotgun，
        再到 Shotgun + Precision 的迁移。
    """
    weapons = weapon_stats[weapon_stats['source_scope'] == 'overview'].copy()
    for col in ['kills', 'damage', 'knockdowns', 'times_knocked', 'playtime_seconds', 'fwr_pct']:
        if col in weapons.columns:
            weapons[col] = to_number(weapons[col])
    weapons['category'] = weapons['weapon'].map(WEAPON_CATEGORY_MAPPING).fillna('Other')

    single_totals = weapons.groupby('event_slug').agg(total_kills=('kills', 'sum'), total_damage=('damage', 'sum')).reset_index()
    single_weapon_share = weapons.merge(single_totals, on='event_slug', how='left')
    single_weapon_share['kill_share_pct'] = 100 * single_weapon_share['kills'] / single_weapon_share['total_kills']
    single_weapon_share['damage_share_pct'] = 100 * single_weapon_share['damage'] / single_weapon_share['total_damage']
    single_weapon_share = add_event_label(single_weapon_share)

    category_source = single_weapon_share[~single_weapon_share['category'].isin(['Ordnance', 'Other'])].copy()
    category_share = category_source.groupby(['event_slug', 'category']).agg(
        kills=('kills', 'sum'),
        damage=('damage', 'sum'),
        playtime_seconds=('playtime_seconds', 'sum'),
    ).reset_index()
    category_totals = category_share.groupby('event_slug').agg(total_kills=('kills', 'sum'), total_damage=('damage', 'sum')).reset_index()
    category_share = category_share.merge(category_totals, on='event_slug', how='left')
    category_share['kill_share_pct'] = 100 * category_share['kills'] / category_share['total_kills']
    category_share['damage_share_pct'] = 100 * category_share['damage'] / category_share['total_damage']
    category_share = add_event_label(category_share)
    return category_share, single_weapon_share


def calc_legend_class_slots(legend_picks: pd.DataFrame, legend_class_mapping: dict[str, str]) -> pd.DataFrame:
    """Calculate legend class slot share by event.

    Input:
        legend_picks: overview legend table with event_slug, legend and pick_rate_pct.
        legend_class_mapping: dictionary mapping legend name to class.

    Output fields:
        event_slug, event_label, class, class_slot_share_pct, occurrences.
        Because each team has three legend slots, class shares should sum close to 300%.

    Report mapping:
        Figure 4 / 图表 4，用于说明职业槽位如何暴露版本需求，
        例如 Championship Support 槽位超过 200%。
    """
    legends = legend_picks.copy()
    legends['pick_rate_pct'] = to_number(legends['pick_rate_pct'])
    legends['total_occurrences'] = to_number(legends['total_occurrences'])
    legends['class'] = legends['legend'].map(legend_class_mapping).fillna('Other')
    class_slots = legends.groupby(['event_slug', 'class']).agg(
        class_slot_share_pct=('pick_rate_pct', 'sum'),
        occurrences=('total_occurrences', 'sum'),
    ).reset_index()
    return add_event_label(class_slots)


def calc_team_style_proxy(match_scores: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Classify team style proxy by kill share and Top 5 conversion.

    Input:
        Preferred input is overview_team_stats, one row per team-event, with games,
        kills, place_pts, total_pts and top_5s. If raw match_scores is passed,
        the function first aggregates by event/team.

    Output fields:
        detail: one row per team-event with kill_share, top5_rate, total_pts_per_game
        and style. summary: event/style aggregates with team count, team_share_pct,
        avg_total_ppg, avg_kill_share and avg_top5_rate.

    Report mapping:
        Figure 5 / 图表 5。This is a proxy for macro style, not a route or POI reconstruction.
    """
    if {'games', 'place_pts', 'total_pts', 'top_5s'}.issubset(match_scores.columns):
        teams = match_scores.copy()
        for col in ['games', 'kills', 'place_pts', 'total_pts', 'top_5s', 'top_10s']:
            teams[col] = to_number(teams[col])
    else:
        scores = match_scores.copy()
        for col in ['kills', 'total_points', 'placement_points_reported', 'placement']:
            scores[col] = to_number(scores[col])
        teams = scores.groupby(['event_slug', 'team']).agg(
            games=('game_id', 'nunique'),
            kills=('kills', 'sum'),
            place_pts=('placement_points_reported', 'sum'),
            total_pts=('total_points', 'sum'),
            top_5s=('placement', lambda values: (values <= 5).sum()),
            top_10s=('placement', lambda values: (values <= 10).sum()),
        ).reset_index()

    teams['kill_share'] = teams['kills'] / teams['total_pts']
    teams['placement_share'] = teams['place_pts'] / teams['total_pts']
    teams['kills_per_game'] = teams['kills'] / teams['games']
    teams['place_pts_per_game'] = teams['place_pts'] / teams['games']
    teams['total_pts_per_game'] = teams['total_pts'] / teams['games']
    teams['top5_rate'] = teams['top_5s'] / teams['games']

    detail_rows = []
    for event_slug, group in teams.groupby('event_slug'):
        kill_median = group['kill_share'].median()
        top5_median = group['top5_rate'].median()
        for _, row in group.iterrows():
            if row['kill_share'] >= kill_median and row['top5_rate'] >= top5_median:
                style = 'hybrid_high_yield'
            elif row['kill_share'] >= kill_median and row['top5_rate'] < top5_median:
                style = 'edge_fighting_proxy'
            elif row['kill_share'] < kill_median and row['top5_rate'] >= top5_median:
                style = 'zone_control_proxy'
            else:
                style = 'low_yield_or_unstable'
            record = row.to_dict()
            record.update({'style': style, 'event_kill_share_median': kill_median, 'event_top5_rate_median': top5_median})
            detail_rows.append(record)

    detail = add_event_label(pd.DataFrame(detail_rows))
    summary = detail.groupby(['event_slug', 'style']).agg(
        teams=('team', 'count'),
        avg_total_ppg=('total_pts_per_game', 'mean'),
        avg_kill_share=('kill_share', 'mean'),
        avg_top5_rate=('top5_rate', 'mean'),
        avg_kills_per_game=('kills_per_game', 'mean'),
        avg_place_ppg=('place_pts_per_game', 'mean'),
    ).reset_index()
    event_team_totals = summary.groupby('event_slug')['teams'].transform('sum')
    summary['team_share_pct'] = 100 * summary['teams'] / event_team_totals
    summary = add_event_label(summary)
    return detail, summary


def classify_concentration(top1: float, top3: float, top5: float) -> str:
    if top1 >= 50 and top3 >= 80 and top5 >= 90:
        return 'single_dominant'
    if top1 >= 50:
        return 'one_core_many_options'
    if top5 >= 80:
        return 'multi_core_narrow_pool'
    if top1 < 25 and top3 < 55:
        return 'open_or_testing'
    return 'moderate_convergence'


def classify_risk_level(row: pd.Series) -> str:
    score = 0
    score += {'single_dominant': 3, 'one_core_many_options': 2, 'multi_core_narrow_pool': 2, 'moderate_convergence': 1, 'open_or_testing': 0}[row['concentration_type']]
    if row['highest_legend_pick_rate'] >= 90:
        score += 3
    elif row['highest_legend_pick_rate'] >= 80:
        score += 2
    elif row['highest_legend_pick_rate'] >= 65:
        score += 1
    if row['highest_class_slot_share'] >= 180:
        score += 3
    elif row['highest_class_slot_share'] >= 150:
        score += 2
    elif row['highest_class_slot_share'] >= 90:
        score += 1
    if row['highest_weapon_category_share'] >= 60:
        score += 3
    elif row['highest_weapon_category_share'] >= 50 or row['highest_single_weapon_share'] >= 30:
        score += 2
    elif row['highest_weapon_category_share'] >= 40 or row['highest_single_weapon_share'] >= 25:
        score += 1
    if row['avg_rez'] >= 15:
        score += 2
    elif row['avg_rez'] >= 10 or row['avg_rspn'] >= 3.5:
        score += 1
    if score >= 10:
        return 'critical'
    if score >= 7:
        return 'high'
    if score >= 4:
        return 'medium'
    return 'low'


def calc_risk_flags(
    score_structure: pd.DataFrame,
    reset_metrics: pd.DataFrame,
    composition_concentration: pd.DataFrame,
    legend_picks: pd.DataFrame,
    class_slots: pd.DataFrame,
    weapon_category_share: pd.DataFrame,
    single_weapon_share: pd.DataFrame,
) -> pd.DataFrame:
    """Generate portfolio risk flags by event.

    Input:
        The event-level outputs from score, reset, composition, legend class and
        weapon share functions.

    Output fields:
        event, top1_comp_share, top3_comp_share, top5_comp_share,
        highest_legend_pick_rate, highest_class_slot_share,
        highest_weapon_category_share, highest_single_weapon_share,
        avg_rez, avg_rspn, risk_level, risk_notes.

    Report mapping:
        Section 6 / 第三层指标系统 and 第四层运营动作。This is a portfolio
        judgment rule, not an official ALGS or Respawn standard.
    """
    legends = legend_picks.copy()
    legends['pick_rate_pct'] = to_number(legends['pick_rate_pct'])
    rows = []
    for event_slug in EVENT_ORDER:
        comp_row = composition_concentration[composition_concentration['event_slug'] == event_slug].iloc[0]
        event_legends = legends[legends['event_slug'] == event_slug].sort_values('pick_rate_pct', ascending=False)
        event_classes = class_slots[class_slots['event_slug'] == event_slug].sort_values('class_slot_share_pct', ascending=False)
        event_weapon_categories = weapon_category_share[
            (weapon_category_share['event_slug'] == event_slug)
            & (~weapon_category_share['category'].isin(['Pistol/SMG', 'Pistol/Akimbo', 'Ordnance', 'Other']))
        ].sort_values('kill_share_pct', ascending=False)
        event_weapons = single_weapon_share[
            (single_weapon_share['event_slug'] == event_slug)
            & (~single_weapon_share['category'].isin(['Ordnance', 'Other']))
        ].sort_values('kill_share_pct', ascending=False)
        reset_row = reset_metrics[reset_metrics['event_slug'] == event_slug].iloc[0]
        top_legend = event_legends.iloc[0]
        top_class = event_classes.iloc[0]
        top_category = event_weapon_categories.iloc[0]
        top_weapon = event_weapons.iloc[0]
        concentration_type = classify_concentration(comp_row['top1_comp_share'], comp_row['top3_comp_share'], comp_row['top5_comp_share'])
        row = {
            'event': EVENT_LABEL[event_slug],
            'event_slug': event_slug,
            'top1_comp_share': comp_row['top1_comp_share'],
            'top3_comp_share': comp_row['top3_comp_share'],
            'top5_comp_share': comp_row['top5_comp_share'],
            'concentration_type': concentration_type,
            'highest_legend': top_legend['legend'],
            'highest_legend_pick_rate': top_legend['pick_rate_pct'],
            'highest_class': top_class['class'],
            'highest_class_slot_share': top_class['class_slot_share_pct'],
            'highest_weapon_category': top_category['category'],
            'highest_weapon_category_share': top_category['kill_share_pct'],
            'highest_single_weapon': top_weapon['weapon'],
            'highest_single_weapon_share': top_weapon['kill_share_pct'],
            'avg_rez': reset_row['avg_rez'],
            'avg_rspn': reset_row['avg_rspn'],
            'deaths_per_game': reset_row['deaths_per_game'],
            'deaths_per_game_minus_60': reset_row['deaths_above_60'],
        }
        row['risk_level'] = classify_risk_level(pd.Series(row))
        if row['risk_level'] in ['critical', 'high']:
            row['risk_notes'] = f"{row['concentration_type']}; {row['highest_class']} {row['highest_class_slot_share']:.1f}%; {row['highest_weapon_category']} {row['highest_weapon_category_share']:.1f}%"
        else:
            row['risk_notes'] = f"{row['concentration_type']}; monitor whether weapon or role concentration becomes durable"
        rows.append(row)
    return pd.DataFrame(rows)


下面这组画图辅助函数也保存在 notebook 中，保证所有图表都能从数据直接生成。为了减少环境依赖，这里使用 `Pillow` 绘图，而不是依赖不可复现的截图。


In [ ]:
FONT_REGULAR_CANDIDATES = [
    '/System/Library/Fonts/Supplemental/Arial.ttf',
    '/System/Library/Fonts/Supplemental/Helvetica.ttf',
    '/System/Library/Fonts/Supplemental/Arial Unicode.ttf',
]
FONT_BOLD_CANDIDATES = [
    '/System/Library/Fonts/Supplemental/Arial Bold.ttf',
    '/System/Library/Fonts/Supplemental/Helvetica Bold.ttf',
    '/System/Library/Fonts/Supplemental/Arial Unicode.ttf',
]

def load_font(size: int, bold: bool = False):
    candidates = FONT_BOLD_CANDIDATES if bold else FONT_REGULAR_CANDIDATES
    for path in candidates:
        if Path(path).exists():
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()


def make_canvas(title: str, subtitle: str = '', size: tuple[int, int] = (1400, 820)):
    image = Image.new('RGB', size, 'white')
    draw = ImageDraw.Draw(image)
    draw.text((72, 42), title, fill='#111827', font=load_font(34, True))
    if subtitle:
        draw.text((72, 92), subtitle, fill='#4B5563', font=load_font(20))
    return image, draw


def map_y(value: float, y0: int, y1: int, max_y: float, min_y: float = 0) -> float:
    if max_y == min_y:
        return y1
    return y1 - ((value - min_y) / (max_y - min_y)) * (y1 - y0)


def draw_axes(draw, x0: int, y0: int, x1: int, y1: int, max_y: float, min_y: float = 0, ticks: int = 5):
    draw.line((x0, y1, x1, y1), fill='#9CA3AF', width=2)
    draw.line((x0, y0, x0, y1), fill='#9CA3AF', width=2)
    for i in range(ticks + 1):
        value = min_y + (max_y - min_y) * i / ticks
        y = map_y(value, y0, y1, max_y, min_y)
        draw.line((x0 - 6, y, x1, y), fill='#EEF2F7', width=1)
        draw.text((x0 - 56, y - 10), f'{value:.0f}', fill='#6B7280', font=load_font(16))


def draw_legend(draw, items: list[tuple[str, str]], x: int, y: int):
    cursor = x
    for label, color in items:
        draw.rounded_rectangle((cursor, y, cursor + 28, y + 16), radius=4, fill=color)
        draw.text((cursor + 38, y - 3), label, fill='#374151', font=load_font(17))
        cursor += 38 + int(draw.textlength(label, font=load_font(17))) + 32


def save_grouped_bar_chart(
    table: pd.DataFrame,
    value_columns: list[tuple[str, str, str]],
    title: str,
    subtitle: str,
    max_y: float,
    output_path: Path,
):
    image, draw = make_canvas(title, subtitle)
    x0, y0, x1, y1 = 120, 172, 1280, 675
    draw_axes(draw, x0, y0, x1, y1, max_y)
    labels = table['event_label'].tolist()
    group_gap = (x1 - x0) / len(labels)
    bar_w = 48
    inner_gap = 18
    group_w = len(value_columns) * bar_w + (len(value_columns) - 1) * inner_gap
    for i, label in enumerate(labels):
        cx = x0 + group_gap * (i + 0.5)
        for j, (column, series_label, color) in enumerate(value_columns):
            value = float(table.iloc[i][column])
            bar_x = cx - group_w / 2 + j * (bar_w + inner_gap)
            bar_y = map_y(value, y0, y1, max_y)
            draw.rounded_rectangle((bar_x, bar_y, bar_x + bar_w, y1), radius=7, fill=color)
            draw.text((bar_x + bar_w / 2, bar_y - 16), f'{value:.0f}', fill='#374151', font=load_font(15, True), anchor='mm')
        draw.text((cx, y1 + 28), label, fill='#111827', font=load_font(19, True), anchor='mm')
    draw_legend(draw, [(label, color) for _, label, color in value_columns], 120, 720)
    image.save(output_path, quality=95)


def save_stacked_bar_chart(
    pivot: pd.DataFrame,
    colors: dict[str, str],
    title: str,
    subtitle: str,
    max_y: float,
    output_path: Path,
    value_fmt: str = '{:.0f}%',
):
    image, draw = make_canvas(title, subtitle)
    x0, y0, x1, y1 = 120, 172, 1280, 675
    draw_axes(draw, x0, y0, x1, y1, max_y)
    labels = [EVENT_LABEL[event] for event in pivot.index]
    gap = (x1 - x0) / len(labels)
    bar_w = 150
    for i, event_slug in enumerate(pivot.index):
        cx = x0 + gap * (i + 0.5)
        bottom = y1
        for category in pivot.columns:
            value = float(pivot.loc[event_slug, category])
            height = (value / max_y) * (y1 - y0)
            if height > 0:
                draw.rectangle((cx - bar_w / 2, bottom - height, cx + bar_w / 2, bottom), fill=colors.get(category, '#9CA3AF'))
                if height >= 36:
                    draw.text((cx, bottom - height / 2), f'{category}\n{value_fmt.format(value)}', fill='white', font=load_font(14, True), anchor='mm', align='center')
            bottom -= height
        draw.rectangle((cx - bar_w / 2, y0, cx + bar_w / 2, y1), outline='#E5E7EB', width=1)
        draw.text((cx, y1 + 28), labels[i], fill='#111827', font=load_font(19, True), anchor='mm')
    draw_legend(draw, [(category, colors.get(category, '#9CA3AF')) for category in pivot.columns], 120, 720)
    image.save(output_path, quality=95)


def save_line_chart(
    table: pd.DataFrame,
    series: list[tuple[str, str, str]],
    title: str,
    subtitle: str,
    max_y: float,
    output_path: Path,
    min_y: float = 0,
):
    image, draw = make_canvas(title, subtitle)
    x0, y0, x1, y1 = 120, 172, 1280, 675
    draw_axes(draw, x0, y0, x1, y1, max_y, min_y=min_y)
    labels = table['event_label'].tolist()
    xs = [x0 + (x1 - x0) * (i + 0.5) / len(labels) for i in range(len(labels))]
    for label, column, color in series:
        points = [(xs[i], map_y(float(table.iloc[i][column]), y0, y1, max_y, min_y), float(table.iloc[i][column])) for i in range(len(table))]
        for a, b in zip(points, points[1:]):
            draw.line((a[0], a[1], b[0], b[1]), fill=color, width=5)
        for x, y, value in points:
            draw.ellipse((x - 9, y - 9, x + 9, y + 9), fill=color, outline='white', width=3)
            draw.text((x, y - 28), f'{value:.1f}', fill=color, font=load_font(15, True), anchor='mm')
    for x, label in zip(xs, labels):
        draw.text((x, y1 + 28), label, fill='#111827', font=load_font(19, True), anchor='mm')
    draw_legend(draw, [(label, color) for label, _, color in series], 120, 720)
    image.save(output_path, quality=95)


## 4. Figure 1: Score and reset signal dashboard

这张图对应报告图表 1。它用 `match_scores.csv` 计算击杀分占比和场均计分击杀，用 `game_team_stats.csv` 计算死亡、救起和重生。

它验证的不是“复活一定变强”，而是更克制的结论：**总得分结构很稳定，但 Split 2 和 Championship 出现了更明显的 reset / second-chance 信号**。


In [ ]:
score_structure = calc_score_structure(match_scores)
reset_metrics = calc_reset_metrics(game_team_stats)

table1_score_reset = score_structure.merge(
    reset_metrics[['event_slug', 'deaths_per_game', 'avg_rez', 'avg_rspn', 'knocks_per_game', 'deaths_above_60']],
    on='event_slug',
    how='left',
)
table1_score_reset = add_event_label(table1_score_reset.drop(columns=['event_label']))
table1_score_reset.to_csv(TABLE_DIR / 'table1_score_reset.csv', index=False)
table1_score_reset.to_csv(INTERMEDIATE_DIR / 'score_reset_metrics_intermediate.csv', index=False)

# Figure 1 combines stable score structure with reset signals.
image, draw = make_canvas(
    'Score Stayed Stable, Reset Signals Rose',
    'Kill-point share stayed near 51%-52%; later metas created more revive/respawn windows.'
)
left = (72, 164, 674, 610)
right = (742, 164, 1328, 610)

def draw_panel(box, title):
    x0, y0, x1, y1 = box
    draw.rounded_rectangle((x0, y0, x1, y1), radius=18, fill='#F9FAFB', outline='#E5E7EB', width=2)
    draw.text((x0 + 24, y0 + 24), title, fill='#111827', font=load_font(22, True))

def draw_small_line(box, data, series, min_y, max_y):
    x0, y0, x1, y1 = box
    plot_x0, plot_y0, plot_x1, plot_y1 = x0 + 76, y0 + 94, x1 - 36, y1 - 66
    for tick in [min_y, (min_y + max_y) / 2, max_y]:
        y = map_y(tick, plot_y0, plot_y1, max_y, min_y)
        draw.line((plot_x0, y, plot_x1, y), fill='#E5E7EB', width=1)
        draw.text((x0 + 24, y - 10), f'{tick:.0f}', fill='#9CA3AF', font=load_font(16))
    xs = [plot_x0 + (plot_x1 - plot_x0) * i / (len(data) - 1) for i in range(len(data))]
    for label, column, color in series:
        points = [(xs[i], map_y(float(data.iloc[i][column]), plot_y0, plot_y1, max_y, min_y), float(data.iloc[i][column])) for i in range(len(data))]
        for a, b in zip(points, points[1:]):
            draw.line((a[0], a[1], b[0], b[1]), fill=color, width=5)
        for x, y, value in points:
            draw.ellipse((x - 8, y - 8, x + 8, y + 8), fill=color, outline='white', width=3)
            draw.text((x, y - 24), f'{value:.1f}', fill=color, font=load_font(15, True), anchor='mm')
    for x, label in zip(xs, data['event_label']):
        draw.text((x, plot_y1 + 28), label, fill='#111827', font=load_font(18, True), anchor='mm')

def draw_small_bars(box, data):
    x0, y0, x1, y1 = box
    plot_x0, plot_y0, plot_x1, plot_y1 = x0 + 76, y0 + 94, x1 - 36, y1 - 66
    for tick in [0, 10, 20]:
        y = map_y(tick, plot_y0, plot_y1, 22)
        draw.line((plot_x0, y, plot_x1, y), fill='#E5E7EB', width=1)
        draw.text((x0 + 24, y - 10), f'{tick:.0f}', fill='#9CA3AF', font=load_font(16))
    group_gap = (plot_x1 - plot_x0) / len(data)
    for i, (_, row) in enumerate(data.iterrows()):
        cx = plot_x0 + group_gap * (i + 0.5)
        for offset, column, color in [(-23, 'avg_rspn', '#F59E0B'), (23, 'avg_rez', '#10B981')]:
            value = float(row[column])
            y = map_y(value, plot_y0, plot_y1, 22)
            draw.rounded_rectangle((cx + offset - 17, y, cx + offset + 17, plot_y1), radius=6, fill=color)
            draw.text((cx + offset, y - 16), f'{value:.1f}', fill=color, font=load_font(15, True), anchor='mm')
        draw.text((cx, plot_y1 + 28), row['event_label'], fill='#111827', font=load_font(18, True), anchor='mm')

draw_panel(left, 'A. Kills and deaths per game')
draw_panel(right, 'B. Reset indicators per game')
draw_small_line(left, table1_score_reset, [('Score kills', 'kills_per_game', '#2563EB'), ('Deaths', 'deaths_per_game', '#EF4444')], 54, 62)
draw_small_bars(right, table1_score_reset)
draw_legend(draw, [('Respawn (rspn)', '#F59E0B'), ('Revive (rez)', '#10B981')], 820, 222)
draw.text((72, 650), 'Readout:', fill='#111827', font=load_font(21, True))
draw.text((172, 650), 'Score share stayed stable, while Championship shows a sharp revive signal.', fill='#374151', font=load_font(19))
draw.text((72, 686), 'Data: match_scores.csv for credited kills; game_team_stats.csv for deaths, rez and rspn.', fill='#6B7280', font=load_font(17))
image.save(FIGURE_DIR / 'fig1_score_reset_dashboard.png', quality=95)

display(table1_score_reset.round(3))


## 5. Figure 2: Composition concentration dashboard

Top 1 / Top 3 / Top 5 覆盖率的口径：

- **Top 1 覆盖率**：该赛事中使用次数最多的一套三人阵容，占全部“队伍-单局阵容记录”的比例。
- **Top 3 覆盖率**：使用次数最多的前三套阵容合计比例。
- **Top 5 覆盖率**：使用次数最多的前五套阵容合计比例。

这组指标看的是职业赛场有没有重新收敛出“标准答案”。


In [ ]:
composition_concentration = calc_composition_concentration(overview_composition_meta)
composition_concentration.to_csv(TABLE_DIR / 'table2_composition_concentration.csv', index=False)
composition_concentration.to_csv(INTERMEDIATE_DIR / 'composition_concentration_intermediate.csv', index=False)

save_grouped_bar_chart(
    composition_concentration,
    [
        ('top1_comp_share', 'Top 1 comp', '#2563EB'),
        ('top3_comp_share', 'Top 3 comps', '#10B981'),
        ('top5_comp_share', 'Top 5 comps', '#F59E0B'),
    ],
    'Composition Concentration Rebuilt After Every Patch',
    'The meta was loosened at EWC, then re-converged sharply by Championship.',
    100,
    FIGURE_DIR / 'fig2_composition_concentration.png',
)

display(composition_concentration.round(2))


## 6. Figure 3: Weapon shift dashboard

这张图计算武器类别的击杀占比，同时额外导出单武器击杀占比榜，用来核对报告中的 HAVOC、Hemlok、Mozambique Akimbo、Shotgun 等结论。

注意：武器变化不一定只来自武器本体。角色阵容会改变交战距离，例如 Gibraltar / Newcastle / Rampart 体系会把更多决战压到近距离，从而放大 Shotgun 的价值。


In [ ]:
weapon_category_share, single_weapon_share = calc_weapon_share(weapon_stats)
weapon_category_share.to_csv(TABLE_DIR / 'table3_weapon_share.csv', index=False)
single_weapon_leaderboard = single_weapon_share[
    ~single_weapon_share['category'].isin(['Ordnance', 'Other'])
].sort_values(['event_slug', 'kill_share_pct'], ascending=[True, False])
single_weapon_leaderboard.to_csv(TABLE_DIR / 'table3_single_weapon_leaderboard.csv', index=False)
weapon_category_share.to_csv(INTERMEDIATE_DIR / 'weapon_category_share_intermediate.csv', index=False)

weapon_categories = ['AR', 'SMG', 'Precision', 'Shotgun', 'Shotgun-Akimbo', 'LMG']
weapon_colors = {
    'AR': '#2563EB', 'SMG': '#7C3AED', 'Precision': '#0891B2', 'Shotgun': '#DC2626',
    'Shotgun-Akimbo': '#F97316', 'LMG': '#16A34A'
}
weapon_pivot = (
    weapon_category_share[weapon_category_share['category'].isin(weapon_categories)]
    .pivot_table(index='event_slug', columns='category', values='kill_share_pct', aggfunc='sum')
    .reindex(EVENT_ORDER)
    .reindex(columns=weapon_categories)
    .fillna(0)
)
save_stacked_bar_chart(
    weapon_pivot,
    weapon_colors,
    'Weapon Kill Share Shifted Between Weapon Ecosystems',
    'AR dominance gave way to Akimbo/Shotgun, then to Shotgun + Precision at Championship.',
    100,
    FIGURE_DIR / 'fig3_weapon_shift.png',
)

display(weapon_category_share[weapon_category_share['category'].isin(weapon_categories)].round(2))
display(single_weapon_leaderboard.groupby('event_label').head(8)[['event_label', 'weapon', 'category', 'kills', 'kill_share_pct', 'damage_share_pct']].round(2))


## 7. Figure 4: Legend class slots dashboard

角色职业槽位按单角色 pick rate 汇总。因为每队每局有 3 个 legend 槽位，所以每次赛事所有职业类别加起来理论上接近 **300%**。

例如 Championship 的 Support 超过 200%，含义不是“Support 队伍超过 200%”，而是平均每支队伍阵容里有约两个 Support 槽位。


In [ ]:
class_slots = calc_legend_class_slots(overview_legend_meta, LEGEND_CLASS_MAPPING)
class_slots.to_csv(TABLE_DIR / 'table4_class_slots.csv', index=False)
class_slots.to_csv(INTERMEDIATE_DIR / 'legend_class_slots_intermediate.csv', index=False)

class_order = ['Assault', 'Controller', 'Recon', 'Support', 'Skirmisher', 'Other']
class_colors = {
    'Assault': '#2563EB', 'Controller': '#10B981', 'Recon': '#8B5CF6',
    'Support': '#F97316', 'Skirmisher': '#06B6D4', 'Other': '#9CA3AF'
}
class_pivot = (
    class_slots.pivot_table(index='event_slug', columns='class', values='class_slot_share_pct', aggfunc='sum')
    .reindex(EVENT_ORDER)
    .reindex(columns=class_order)
    .fillna(0)
)
save_stacked_bar_chart(
    class_pivot,
    class_colors,
    'Legend Class Slots Show What the Meta Needed',
    'Class shares sum three legend slots per team; 200% Support means roughly two Support slots per team.',
    300,
    FIGURE_DIR / 'fig4_legend_class_slots.png',
)

slot_total_check = class_slots.groupby('event_slug', as_index=False)['class_slot_share_pct'].sum().rename(columns={'class_slot_share_pct': 'slot_total_pct'})
slot_total_check = add_event_label(slot_total_check)
display(class_slots.round(2))
display(slot_total_check.round(2))


## 8. Figure 5: Team style proxy

这里把每次赛事中的队伍按两个维度分成四类：

- 队伍击杀分占比是否高于赛事内中位数。
- 队伍 Top 5 率是否高于赛事内中位数。

四类分别是：

- `hybrid_high_yield`：击杀占比高，Top 5 转化也高。
- `edge_fighting_proxy`：击杀占比高，但 Top 5 转化低。
- `zone_control_proxy`：击杀占比低，但 Top 5 转化高。
- `low_yield_or_unstable`：击杀占比和 Top 5 转化都低。

这只是一个 **proxy**，不是路线还原。公开数据没有完整坐标和路线，因此不能直接说某队每局一定是“早进圈”或“圈边”。


In [ ]:
team_style_detail, team_style_summary = calc_team_style_proxy(overview_team_stats)
team_style_detail.to_csv(INTERMEDIATE_DIR / 'team_style_proxy_detail.csv', index=False)
team_style_summary.to_csv(TABLE_DIR / 'table5_team_style_ppg.csv', index=False)

style_order = ['edge_fighting_proxy', 'hybrid_high_yield', 'zone_control_proxy', 'low_yield_or_unstable']
style_labels = {
    'edge_fighting_proxy': 'Edge',
    'hybrid_high_yield': 'Hybrid',
    'zone_control_proxy': 'Zone',
    'low_yield_or_unstable': 'Low/unstable',
}
style_colors = {
    'edge_fighting_proxy': '#DC2626',
    'hybrid_high_yield': '#2563EB',
    'zone_control_proxy': '#10B981',
    'low_yield_or_unstable': '#9CA3AF',
}
style_plot = team_style_summary.pivot_table(index='event_slug', columns='style', values='avg_total_ppg', aggfunc='mean').reindex(EVENT_ORDER).reset_index()
style_plot['event_label'] = style_plot['event_slug'].map(EVENT_LABEL)
save_line_chart(
    style_plot,
    [(style_labels[style], style, style_colors[style]) for style in style_order],
    'Pure Edge Fighting Scored Less Than Hybrid or Zone Styles',
    'High-kill games mattered, but the best yield came from turning fights into Top 5 conversions.',
    8,
    FIGURE_DIR / 'fig5_team_style_ppg.png',
)

display(team_style_summary.round(3))
display(team_style_detail[['event_label', 'team', 'games', 'kill_share', 'top5_rate', 'total_pts_per_game', 'style']].round(3).head(20))


## 9. Section 6 risk flags

这一节把第三层指标系统落成一张 `risk_flags_by_event.csv`。它用于作品集里的运营判断，不是官方标准。

风险等级是人工规则：阵容集中、最高角色 pick、最高职业槽位、最高武器类别/单武器占比、reset 信号共同越高，风险等级越高。它的作用是帮助运营决定“观察、预警、调优、赛事规则讨论”的优先级。


In [ ]:
risk_flags_by_event = calc_risk_flags(
    score_structure=score_structure,
    reset_metrics=reset_metrics,
    composition_concentration=composition_concentration,
    legend_picks=overview_legend_meta,
    class_slots=class_slots,
    weapon_category_share=weapon_category_share,
    single_weapon_share=single_weapon_share,
)
risk_flags_by_event.to_csv(TABLE_DIR / 'risk_flags_by_event.csv', index=False)

display(risk_flags_by_event.round(2))


## 10. Export all outputs

最后检查所有报告表格和图表是否已经生成。这个 checklist 也是交付包自查的一部分。


In [ ]:
expected_tables = [
    'data_quality_summary.csv',
    'table1_score_reset.csv',
    'table2_composition_concentration.csv',
    'table3_weapon_share.csv',
    'table3_single_weapon_leaderboard.csv',
    'table4_class_slots.csv',
    'table5_team_style_ppg.csv',
    'risk_flags_by_event.csv',
]
expected_figures = [
    'fig1_score_reset_dashboard.png',
    'fig2_composition_concentration.png',
    'fig3_weapon_shift.png',
    'fig4_legend_class_slots.png',
    'fig5_team_style_ppg.png',
]
expected_intermediate = [
    'match_scores_clean.csv',
    'game_composition_quality_check.csv',
    'team_rows_per_game.csv',
    'reset_metrics_by_game.csv',
    'score_reset_metrics_intermediate.csv',
    'composition_concentration_intermediate.csv',
    'weapon_category_share_intermediate.csv',
    'legend_class_slots_intermediate.csv',
    'team_style_proxy_detail.csv',
]

checklist = []
for name in expected_tables:
    path = TABLE_DIR / name
    checklist.append({'type': 'table', 'file': name, 'exists': path.exists(), 'path': str(path.relative_to(OUTPUT_DIR))})
for name in expected_figures:
    path = FIGURE_DIR / name
    checklist.append({'type': 'figure', 'file': name, 'exists': path.exists(), 'path': str(path.relative_to(OUTPUT_DIR))})
for name in expected_intermediate:
    path = INTERMEDIATE_DIR / name
    checklist.append({'type': 'intermediate', 'file': name, 'exists': path.exists(), 'path': str(path.relative_to(OUTPUT_DIR))})

output_checklist = pd.DataFrame(checklist)
output_checklist.to_csv(TABLE_DIR / 'output_checklist.csv', index=False)
display(output_checklist)

print('All expected outputs generated:', bool(output_checklist['exists'].all()))
print('Tables:', TABLE_DIR)
print('Figures:', FIGURE_DIR)
print('Intermediate:', INTERMEDIATE_DIR)
